In [1]:
from typing import Annotated, List, TypedDict, Union, Optional, Literal
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
import os, sys

c:\Users\user\potenup\ReNe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(project_root)

In [5]:
try:
    from src.core.database import SessionLocal
    from src.models.interview import ReneInterview, TrialsReneDetail
except ImportError:
    print("DB 모듈 임포트 실패: DB 저장 단계는 Mock으로 대체됩니다.")
    SessionLocal = None
    ReneInterview = None
    TrialsReneDetail = None

print(f"Project Root added: {project_root}")

Project Root added: c:\Users\user\potenup\ReNe


In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# 그래프 상태 정의
class InterviewState(TypedDict):
    # 대화 기록 (Langraph가 자동으로 append 처리)
    messages: Annotated[List[BaseMessage], add_messages] # 대화 기록
    status: Literal["interview", "evaluate", "analyze", "done"]
    current_turn: int # 현재 턴수
    red_flag_count: int # 연속 결격 횟수(조기 종료용)
    stop_signal: bool # 강제 종료 플래그

    # RAG 검색용 메타데이터
    jobseeker_id: int # 구직자 ID
    company_id: int # 기업 ID
    job_group_id: int # 직군 ID
    company_info: dict # ChromaDB에서 가져온 기업 정보
    jobseeker_info: dict # ChromaDB에서 가져온 구직자 정보
    resume_context: str
    portfolio_context: str
    company_introduction_context: str
    recruitment_notice_context: str

    # 평가 데이터 (DB 저장용)
    evaluation_history: List[dict]

    # 최종 리포트 데이터 (Analyst 결과)
    final_report: Optional[dict] # Analyst 결과
    db_payload: Optional[dict] # DB 삽입용 정리된 데이터  


In [8]:
def load_prompt_markdown(filename: str) -> str:
    """src/prompts 디렉토리에서 마크다운 프롬프트 파일을 읽어옵니다."""
    # 경로: ReNe/src/prompts/filename
    path = os.path.join(project_root, 'src', 'prompts', filename)
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print(f"프롬프트 파일을 찾을 수 없습니다.: {path}")

print(len(load_prompt_markdown("Corporate_Recruiter.md")))

 

1039


In [9]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

In [ ]:
from src.repositories.resume_repository.resume_repository import ResumeRepository
from src.repositories.portfolio_repository.portfolio_repository import PortfolioRepository
from src.repositories.company_repository.company_repository import CompanyRepository
from src.repositories.recruitment_notice_repository.recruitment_notice_repository import RecruitmentNoticeRepository
from src.repositories.company_introduction_repository.company_introduction_repository import CompanyIntroductionRepository
from src.repositories.jobseeker_repository.jobseeker_repository import JobseekerRepository


In [ ]:
def initialize_node(state: InterviewState):
    print("--- [Initialize] 데이터 전체 로딩 ---")

    # DB에서 텍스트 전체를 가져옵니다.
    resume_text = ResumeRepository.get_full_text(state["jobseeker_id"])
    portfolio_text = PortfolioRepository.get_full_text(state["company_id"])
    company_info = CompanyRepository.get_info_by_id(state["company_id"])
    jobseeker_info = JobseekerRepository.get_info_by_id(state["jobseeker_id"])
    company_introduction = CompanyIntroductionRepository.get_full_text(state["company_id"])
    recruitment_notice = RecruitmentNoticeRepository.get_full_text(state["job_group_id"])





In [ ]:
def interviewer_node(state: InterviewState) -> InterviewState:
    print("\n [Interview] 질문 생성 중")
    """
    이전 평가 결과와 면접 진행 상황에 맞춰 다음 질문을 생성합니다.
    """
    system_prompt = load_prompt_markdown("ReNe_of_Trials.md")

    fail_count = state.get("fail_count", 0)
    advice = ""

    if fail_count > 0:
        advice = "(주의: 지원자가 이전 질문에 답변을 잘 못했습니다. 조금 더 기초적인 질문이나 힌트를 주세요.)"

    
    
    


In [ ]:
def evaluator_node(state: InterviewState) -> InterviewState:
    

In [ ]:
def analyzer_node(state: InterviewState) -> InterviewState:
    